In [1]:
import pickle 
import numpy as np 
import itertools
import pandas as pd
from pathlib import Path
import os

### Make manifest to evaluate alternate architectures on Popham conditions

In [8]:
archs_to_skip = [0, 3, 5] # arch configs that didn't train (too large, didn't converge, etc.)
path_to_arch_configs = [config for config in Path("config/arch_search").glob('*v10*.yaml') 
                        if not any([f'_{i}' in config.stem for i in archs_to_skip])]


### Get checkpoints 
# get latest checkpoint for each config 
ckpt_dir = Path('attn_cue_models')

ckpt_dict = {}
for config in path_to_arch_configs:
    ckpt_files = list(ckpt_dir.glob(f'{config.stem}/checkpoints/*.ckpt'))
    if len(ckpt_files) == 0:
        # print(f"Warning: no ckpt found for {config.stem}")
        continue
    latest_ckpt = max(ckpt_files, key=lambda x: x.stat().st_mtime)
    ckpt_dict[config.stem] = latest_ckpt
    print(latest_ckpt)

## Get good configs 
path_to_arch_configs = [config for config in path_to_arch_configs if config.stem in ckpt_dict]

print("N archs to test:", len(path_to_arch_configs))
print("Configs (as posix paths)")
print("\n".join( str(path) for path in path_to_arch_configs))

test_condition_ixs = np.arange(0,12) # 12 test conditions

condition_mapping = itertools.product(path_to_arch_configs, test_condition_ixs)


attn_cue_models/word_task_v10_4MGB_ln_first_arch_1/checkpoints/epoch=2-step=44750-v4.ckpt
attn_cue_models/word_task_v10_4MGB_ln_first_arch_10/checkpoints/epoch=2-step=33358-v6.ckpt
attn_cue_models/word_task_v10_4MGB_ln_first_arch_12/checkpoints/epoch=2-step=33358-v4.ckpt
attn_cue_models/word_task_v10_4MGB_ln_first_arch_2/checkpoints/epoch=3-step=50037-v3.ckpt
attn_cue_models/word_task_v10_4MGB_ln_first_arch_4/checkpoints/epoch=1-step=34375-v3.ckpt
attn_cue_models/word_task_v10_4MGB_ln_first_arch_6/checkpoints/epoch=3-step=42037.ckpt
attn_cue_models/word_task_v10_4MGB_ln_first_arch_7/checkpoints/epoch=0-step=8000-v3.ckpt
attn_cue_models/word_task_v10_4MGB_ln_first_arch_8/checkpoints/epoch=4-step=54716-v4.ckpt
attn_cue_models/word_task_v10_4MGB_ln_first_arch_9/checkpoints/epoch=1-step=16679-v4.ckpt
N archs to test: 9
Configs (as posix paths)
config/arch_search/word_task_v10_4MGB_ln_first_arch_1.yaml
config/arch_search/word_task_v10_4MGB_ln_first_arch_10.yaml
config/arch_search/word_task_

In [9]:
test_condition_manifest = {}
for ix, (config, test_idx) in enumerate(condition_mapping):
    test_condition_manifest[ix] = {
        'config_path': config,
        'ckpt_path': ckpt_dict[config.stem],
        'test_idx': test_idx
    }


In [10]:
outdir = Path("swc_test_manifests")
outdir.mkdir(exist_ok=True, parents=True)

with open(outdir / "arch_search_configs_v10_all_popham_conds_w_latest_ckpts.pkl", 'wb') as f:
    pickle.dump(test_condition_manifest, f, protocol=pickle.HIGHEST_PROTOCOL)
#     match_human_pilot_conds = pickle.load(f)

In [11]:
isinstance(test_condition_manifest[1], dict)

True

In [12]:
### Print n tests for job array ix size 

n_tests = len(test_condition_manifest)
print(f"N tests: {n_tests}")
print(f"Array_ids: 0-{n_tests-1}")


N tests: 108
Array_ids: 0-107
